# Day05 下午个人项目：电商用户多维分析

**姓名：** Steve-cza  
**专题方向：** A

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# 个人信息与专题
# =========================
STUDENT_NAME = "Steve-cza"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)


In [ ]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

### 检查点0完成标志

- [x] 已填写姓名；
- [x] `TOPIC`只填写A、B、C、D或E；
- [x] 输出目录为`output/day05_analysis`；
- [x] Notebook文件名保持为`day05_pm_student_project.ipynb`。


## 任务1：读取并验收数据（必做）

In [ ]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

In [ ]:
# 核心字段与数据验收
core_cols = [
    "CustomerID", "Churn", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder",
]

# 数据验收表
validation = pd.DataFrame({
    "验收项": [
        "数据形状",
        "CustomerID重复数",
        "核心字段缺失数",
        "Churn取值",
    ],
    "期望结果": ["(5630, 22)", 0, 0, "{0, 1}"],
    "实际结果": [
        str(df.shape),
        df["CustomerID"].duplicated().sum(),
        df[core_cols].isna().sum().sum(),
        str(set(df["Churn"].dropna().unique())),
    ],
})

display(validation)


In [ ]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

### 数据粒度说明

请用一句话说明一行数据代表什么：

> 一行数据代表一名注册电商平台的独立用户，不是一笔订单。数据包含该用户的个人信息、行为记录和流失标签。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> CustomerID是用户唯一标识符，属于名义变量（ID编号），数值本身无数学意义（如编号50001和50002的平均值无业务含义），不适合作为连续数值计算均值。


## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [ ]:
# 公共基础指标
overall_metrics = pd.DataFrame({
    "指标": [
        "用户数", "流失人数", "总体流失率",
        "平均订单数", "订单数中位数", "平均优惠券使用次数",
        "平均返现", "平均App使用时长", "平均满意度",
        "平均距上次下单天数",
    ],
    "数值": [
        len(df),
        int(df["Churn"].sum()),
        round(df["Churn"].mean(), 6),
        round(df["OrderCount"].mean(), 2),
        int(df["OrderCount"].median()),
        round(df["CouponUsed"].mean(), 2),
        round(df["CashbackAmount"].mean(), 2),
        round(df["HourSpendOnApp"].mean(), 2),
        round(df["SatisfactionScore"].mean(), 2),
        round(df["DaySinceLastOrder"].mean(), 2),
    ],
})

display(overall_metrics)


In [ ]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

overall_churn_rate = df["Churn"].mean()

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> 当前样本共有5630名电商用户，总体流失率为16.84%（948人），平均订单数为2.96次，平均返现金额为177.22元，平均满意度为3.07分（满分5分）。


## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [ ]:
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])

# 主分组字段
segment_field = "TenureGroup"

# groupby + agg 命名聚合
segment_analysis = (
    df.groupby(segment_field, observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失率=("Churn", "mean"),
          流失人数=("Churn", "sum"),
          平均订单数=("OrderCount", "mean"),
          平均返现=("CashbackAmount", "mean"),
      )
      .reset_index()
)

# 按业务意义排序
segment_analysis[segment_field] = pd.Categorical(
    segment_analysis[segment_field], categories=TENURE_ORDER, ordered=True
)
segment_analysis = segment_analysis.sort_values(segment_field).reset_index(drop=True)

display(segment_analysis)


In [ ]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

### 单维专题分析记录

**本专题要回答的业务问题：**

> 不同生命周期阶段（新用户、0-6个月、7-12个月、13-24个月、24个月以上）用户的流失率、平均订单数和平均返现是否存在差异？

**数据现象：**

> 新用户群体（508人）的流失率最高（53.5%），平均订单数仅1.89次；而24个月以上用户（429人）流失率为0%，平均订单数达3.55次。从新用户到长期用户，流失率呈持续下降趋势，订单数和返现金额逐步上升。

**可能解释：**

> 生命周期阶段与流失率存在明显的负相关关系。这可能是因为长期用户已形成稳定的使用习惯和粘性，但需注意：当前样本仅反映某一时点的截面数据，无法排除用户自身属性（如支付能力、需求频率）等混杂因素。究竟是时长本身促进了留存，还是愿意长期使用的用户本身需求更强，仍需结合用户行为序列数据进行验证。


## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [ ]:
# 选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "PreferredPaymentMode"

# 双维交叉分析
cross_analysis = (
    df.groupby([dim_1, dim_2], observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
      )
      .reset_index()
)

# 样本提示
cross_analysis["样本提示"] = np.where(
    cross_analysis["用户数"] < 30, "小样本", "可观察"
)

# 排序
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]
cross_analysis[dim_1] = pd.Categorical(
    cross_analysis[dim_1], categories=TENURE_ORDER, ordered=True
)
cross_analysis = cross_analysis.sort_values([dim_1, dim_2]).reset_index(drop=True)

display(cross_analysis)


In [ ]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

### 双维分析记录

**最值得关注的维度组合：**

> 新用户 + Cash on Delivery（货到付款）：该组合流失率高达72.4%，是新用户中最高的支付方式组合。

**该组合的用户数、流失率和比较对象：**

> 该组合共58人，流失率72.4%。相比新用户 + Credit Card（47.0%）和新用户 + Debit Card（49.4%），货到付款用户流失率显著更高。此外，24个月以上用户中所有支付方式的流失率均为0%，与新用户形成鲜明对比。

**是否存在小样本风险：**

> 仅24个月以上 + UPI组合（14人）标记为"小样本"，用户数少于30。其他组合均可观察，整体分析的基础较为可靠。

**为什么不能直接写成因果结论：**

> 当前数据是截面数据（仅反映某一时间点），分组差异只能说明TenureGroup和PreferredPaymentMode与流失率存在关联，但不能确定是支付方式"导致"了流失。例如，选择货到付款的新用户可能本身风险偏好较低或对新平台信任不足，这些潜在因素同时影响了支付方式选择和流失行为。需要控制用户画像变量或进行A/B实验才能探讨因果关系。


## 任务5：输出统计报表（必做）

In [ ]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

In [ ]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

## 任务6：结论、限制与建议（必做）

### 结论1

在新用户（508人）中，流失率为53.5%，与24个月以上用户（0%）相比差异显著。对应证据表：segment_analysis（单维专题分析表）。

当前样本显示用户生命周期阶段与流失率存在强负相关，新用户群体流失风险最高。这可能与新用户尚未建立平台使用习惯有关，但仍需结合用户活跃度时序数据进一步验证。

### 结论2

在全体用户（5630人）中，平均订单数为2.96次，平均返现金额为177.22元。长期用户（13-24个月和24个月以上）平均订单数（3.70次、3.55次）和平均返现金额（204.92元、222.34元）均显著高于新用户（1.89次、142.44元）。对应证据表：overall_metrics和segment_analysis。

这可能说明随着使用时长增加，用户对平台的信任度和依赖度提高，从而产生更多订单行为。但也不排除高活跃用户本身更倾向于长期留存的可能。

### 结论3

在双维交叉分析中，新用户 + 货到付款组合（58人）流失率高达72.4%，远高于新用户 + 信用卡（47.0%）和所有24个月以上用户（0%）。对应证据表：cross_analysis。

该组合值得重点关注。货到付款在新用户中可能反映对平台的信任度不足，但本次分析无法分离信任因素和支付方式本身的效应，需引入用户问卷或信任度评分进一步验证。

### 分析限制

当前数据为截面数据，缺少用户完整的订单时间序列。因此无法计算严格意义上的复购率（需区分首购和复购事件），也无法判断用户流失前行为变化的动态趋势。此外，数据缺少外部因素（如促销活动、竞品动向、宏观经济）信息，可能影响结论的完整性。

### 运营建议与验证方式

建议：针对新用户群体设计定向运营策略（如首单优惠、新手引导、定期推送），重点降低新用户流失率；对选择货到付款的用户，可考虑引导其绑定信用卡或电子钱包以提升支付便捷度。

验证方式：实施A/B实验，将新用户随机分为实验组（新策略）和对照组（现有策略），追踪30天、60天、90天留存率差异。同时建议补充用户活跃日志数据，用于分析流失前的行为变化信号。


## 拓展任务（选做）

In [ ]:
# 拓展：将双维分析整理为长表，供Day06绘图使用
# 将cross_analysis转换为pivot格式，便于后续可视化
cross_pivot = cross_analysis.pivot_table(
    index="TenureGroup",
    columns="PreferredPaymentMode",
    values="流失率",
    aggfunc="first",
)
display(cross_pivot)


## 最终检查：GitHub提交前验收

In [ ]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

### GitHub提交清单

- [x] 已填写姓名和专题；
- [x] Notebook已重启内核并从头运行成功；
- [x] 所有检查点均已通过；
- [x] `output/day05_analysis/`中包含三个CSV；
- [x] CSV中没有`Unnamed`索引列；
- [x] 至少完成3条结论、1条限制和1项建议；
- [x] 没有把返现写成消费额；
- [x] 没有把相关关系写成确定因果关系；
- [x] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？——生命周期阶段与流失率之间存在强负相关，新用户流失率高达53.5%。
2. 哪个检查点最能帮助你发现错误？——检查点3（单维专题）帮助确认分组用户数之和等于总用户数，避免数据筛选遗漏。
3. 哪条结论最容易被误解为因果关系？——流失率随生命周期阶段下降，容易被误解为"时间越长越不会流失"，但可能是有需求差异的用户主动选择留存。
4. 如果增加一个字段，你最希望增加什么？——希望增加用户注册日期和每次下单时间，用于计算留存曲线和流失前的行为变化。
5. 第6天准备把哪张统计表转化为图表？为什么？——segment_analysis表（TenureGroup分组），因为生命周期阶段非常适合用柱状图和折线图展示流失率差异。
